# GECCO2019 - Bi-objective Traveling Thief Problem 

This worksheet contains the evaluation for the competition at **GECCO2019**. 
After having received all submissions, the evaluation will be done as follows:

After having received all submissions, the evaluation will be done as follows:

For each of the nine test problems

a) We will merge the solution sets of all submissions and extract the non-dominated set.

b) The minimum in time and the maximum in profit will be used to determine the reference point.

c) With respect to this reference point the quality of each submission will be measured using the hypervolume indicator.

d) We will sort the submissions according to the achieved hypervolume in descending order and give points as follows: 1st place -> 3 points, 2nd place -> 2 points, 3rd place -> 1 point.


By adding up the points for each submission we will create the overall ranking. Please note, that depending on the number of submissions the evaluation might need to be reconsidered.

The validation has already been done using the Java code. Which means that each submission has the correct number of solutions (less than the maximum specfied at the competition homepage).


## Imports necessary for the evaluation

In [ ]:
import sys
import os.path
eval = os.path.abspath("gecco19-thief/submissions/!EVALUATION")
sys.path.append(eval)


In [ ]:
from non_dominated_sorting import fast_non_dominated_sort
import numpy as np
import matplotlib.pyplot as plt
from hv import Hypervolume
from normalization import normalize
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import matplotlib.cm as cm
cmap = cm.get_cmap('tab10')

# Handling old numpy deprecations
if not hasattr(np, 'int'):
    np.int = int


## Participants and Problems

In [ ]:
# the result folder as a path
folder = os.path.abspath("gecco19-thief/submissions")
group_5_folder = os.path.abspath("Submissions")


# all submissions received
participants = ["ALLAOUI","jomar", "shisunzhang",  "faria", "HPI",
                "NTGA", "SSteam", "SamirO-ETF-ba", "FRA", "sinc", "JG"
                 # "ValAurTeam", "MicroGA" ## unfortunatly those submissions were invalid  
               ]


# all the problems to be solved
problems = ["a280_n279", "a280_n1395", "a280_n2790",
            "fnl4461_n4460", "fnl4461_n22300", "fnl4461_n44600", 
            "pla33810_n33809", "pla33810_n169045", "pla33810_n338090"
]


## Load data

Load all data from the submission directory and convert it to a minimization problem.
The data dictionary contains all submissions of a participant. The set of non-dominated points
is converted to a minimization problem by multiplying the profit times -1.

In [ ]:

data = {}

for problem in problems:
    _entry = {}
    # Group 5
    fname = "Group_5_%s.f" % (problem)   
    path_to_file = os.path.join(group_5_folder, fname)
    if not os.path.isfile(path_to_file):
        fname = "Group_5_%s.f" % (problem.replace("_", "-"))
        path_to_file = os.path.join(group_5_folder, fname)
    _F = np.loadtxt(path_to_file)
    _entry["Group_5"] = _F * [1, -1]

    # Other teams    
    for participant in participants:
        
        # check for the corresponding file
        fname = "%s_%s.f" % (participant, problem)   
        path_to_file = os.path.join(folder,participant, fname)
        
        # in case the wrong delimiter was used
        if not os.path.isfile(path_to_file):
            fname = "%s_%s.f" % (participant, problem.replace("_", "-"))
            path_to_file = os.path.join(folder,participant, fname)
         
        # load the values in the objective space - first column is time, second profit
        _F = np.loadtxt(path_to_file)
        
        # modify it to a min-min problem by multiplying the profit by -1
        _entry[participant] = _F * [1, -1]
        
    data[problem] = _entry



## Plot the results for Group 5

##### Box plots of time and profit

In [ ]:
team = "Group_5"

# Define the groups based on problem name
groups = {
    "a280": [],
    "fnl4461": [],
    "pla33810": []
}

# Collect data for the team
for problem in problems:
    F = data[problem][team]
    time = F[:, 0]
    profit = -F[:, 1]  # convert back to profit
    for key in groups:
        if key in problem:
            groups[key].append({"problem": problem, "time": time, "profit": profit})

# --- Create separate plots ---
for group_name, group_data in groups.items():
    # Prepare data for boxplots
    times = [d["time"] for d in group_data]
    profits = [d["profit"] for d in group_data]
    labels = [d["problem"] for d in group_data]

    # --- Time plot ---
    plt.figure(figsize=(6, 5))
    plt.boxplot(times, tick_labels=labels, showfliers=False)
    plt.title(f"{group_name} - Time distribution")
    plt.ylabel("Time")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # --- Profit plot ---
    plt.figure(figsize=(6, 5))
    plt.boxplot(profits, tick_labels=labels, showfliers=False)
    plt.title(f"{group_name} - Profit distribution")
    plt.ylabel("Profit")
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()




#### Time vs negative profit graphs

In [ ]:
team = "Group_5"

for k, problem in enumerate(problems):
    _F = data[problem][team]
    time = _F[:, 0]
    neg_profit = _F[:, 1] 

    plt.scatter(time, neg_profit, label=problem, s=10, facecolors='none', edgecolors=cmap(0)) 

    # Compute the non-dominated front for this problem
    I = fast_non_dominated_sort(_F)[0]
    _non_dom = _F[I, :]

    _min = _non_dom.min(axis=0)
    _max = _non_dom.max(axis=0)
    _range = _max - _min

    plt.xlabel("time")
    plt.ylabel("negative profit")
    plt.xlim(_min[0] - 0.05 * _range[0], _max[0] + 0.05 * _range[0])
    plt.ylim(_min[1] - 0.05 * _range[1], _max[1] + 0.05 * _range[1])
    plt.title(f"Group_5: {problem}")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()



## Plot the results for all teams

To get an idea how the submissions look like, we are plotting the results first.
Note that the plots are not normalized yet and the ranges of both object vary.

In [ ]:
participants.append("Group_5")

In [ ]:
print("Plot the results. If points are not shown there are not in the non-dominated region.")

only_top_3 = True

for problem in problems:
    
    for k, participant in enumerate(participants):
        
        if not only_top_3 or (only_top_3 and participant in ["HPI", "jomar", "NTGA", "Group_5"]):

            _F = data[problem][participant]
            plt.scatter(_F[:,0], _F[:,1], label=participant, s=10, facecolors='none', edgecolors=cmap(k))

    _all = np.row_stack([data[problem][participant] for participant in participants])
    I = fast_non_dominated_sort(_all)[0]
    _non_dom = _all[I]
    
    _min = _non_dom.min(axis=0)
    _max = _non_dom.max(axis=0)
    _range = _max - _min
        
    print("=" * 60)
    print(problem)
    print("=" * 60)
    plt.xlabel("time")
    plt.ylabel("negative profit")
    plt.title(f"Top teams and Group 5: {problem}")
    plt.xlim(_min[0] - 0.05 * _range[0], _max[0] + 0.05 * _range[0])
    plt.ylim(_min[1] - 0.05 * _range[1], _max[1] + 0.05 * _range[1])
    plt.legend()
    plt.show()
        

## Find the reference non-dominated set for each test instance

For each problem we merge the submissions to a new population and filter out the non-dominated solutions. Then, we take the minimum and the maximum of this set as the ideal and nadir point the normalize the results.

In [ ]:
# One table of hypervolume results for all problems and participants

import pandas as pd
import numpy as np

# Compute ideal and nadir points and store non-dominated fronts
ideal_point = {}
nadir_point = {}
ndf = {}

for problem in problems:
    # Merge all participants for this problem
    M = []
    for participant in participants:
        _F = data[problem][participant]
        M.append(_F)
    M = np.vstack(M)
    
    # Non-dominated sorting
    I = fast_non_dominated_sort(M)[0]
    M_nd = M[I, :]
    
    ideal_point[problem] = np.min(M_nd, axis=0)
    nadir_point[problem] = np.max(M_nd, axis=0)
    ndf[problem] = M_nd

# Compute hypervolumes for each participant and problem
hv_dict = {participant: {} for participant in participants}

for problem in problems:
    z = ideal_point[problem]
    z_nad = nadir_point[problem]

    for participant in participants:
        _F = data[problem][participant]
        _N = normalize(_F, z, z_nad)
        _hv = Hypervolume(np.array([1,1])).calc(_N)
        hv_dict[participant][problem] = _hv

# Convert to DataFrame: rows = participant, columns = problem, values = HV
hv_table = pd.DataFrame.from_dict(hv_dict, orient='index')
hv_table.index.name = "participant"
hv_table.reset_index(inplace=True)

print(hv_table)


In [ ]:
# Seperate tables for hypervolume results per problem instance

ideal_point = {}
nadir_point = {}
ndf = {}

for problem in problems:
    
    # the merged non-dominated solutions for the specific problem
    M = []
    for participant in participants:    
        _F = data[problem][participant]
        M.append(_F)
        
    M = np.vstack(M)    
    I = fast_non_dominated_sort(M)[0]
    M = M[I, :]
    
    ideal_point[problem] = np.min(M, axis=0)
    nadir_point[problem] = np.max(M, axis=0)
    ndf[problem] = M

results = []

for problem in problems:
    
    z = ideal_point[problem]
    z_nad = nadir_point[problem]
 
    for participant in participants:    
        _F = data[problem][participant]
        _N = normalize(_F, z, z_nad)
        _hv = Hypervolume(np.array([1,1])).calc(_N)
        results.append({'problem' : problem, 'participant' : participant, 'hv' : _hv})
        
df = pd.DataFrame(results, columns=["problem", "participant", "hv"])

for problem in problems:

    print("=" * 60)
    print(problem)
    print("=" * 60)
    
    _df = df[df["problem"] == problem].copy()
    _df.sort_values("hv", ascending=False, inplace=True)
    _df.reset_index(drop=True, inplace=True)
    print(_df)
    

In the following for each problem the non-dominated set of solutions is first normalized using the boundaries and hypervolume is calculated.

The data frame contains all results. Now, we need to rank the submission for each test instance:

In [ ]:
# the final ranking. And add zero points initially (sum is later taken anyway...)
ranking = []
for participant in participants:
    ranking.append({'participant': participant, 'points' : 0})


# one more time loop through problem wise
for problem in problems:
    
    _df = df[df["problem"] == problem].copy()
    
    # sort descending by hv
    _df.sort_values("hv", ascending=False, inplace=True)
    
    # 3 points for the 1st place
    first = _df.iloc[0]["participant"]
    ranking.append({'participant': first, 'points' : 3})
    
    # 2 points for the 2nd place
    second = _df.iloc[1]["participant"]
    ranking.append({'participant': second, 'points' : 2})
    
    # 1 point for the 3rd place
    third = _df.iloc[2]["participant"]
    ranking.append({'participant': third, 'points' : 1})

    
ranking = pd.DataFrame(ranking, columns=["participant", "points"])

# Leaderboard

Finally, we sum up the hypervolume for each problem and evaluate the winner!

In [ ]:
ranking.groupby('participant').sum().sort_values("points", ascending=False)